In [3]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files


uploaded = files.upload()
file_name = next(
    name for name in uploaded if name.lower().endswith(".csv")
)

df = pd.read_csv(file_name)


df = df.drop_duplicates().copy()

numeric_cols = [
    "Age", "GPA", "Attendance_Rate",
    "Study_Hours_per_Week",
    "Current_Semester_Performance",
    "Courses_Failed", "Dropout_Risk"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


df["Dropout_Risk_Label"] = (
    df["Dropout_Risk"]
    .map({0: "Low / No Flag", 1: "High Risk"})
    .fillna("Unknown")
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print(df.columns.tolist())


Saving student_management_dataset (1).csv to student_management_dataset (1) (2).csv
Dataset loaded successfully!
Dataset shape: (1000, 24)
['Student_ID', 'Age', 'Gender', 'Ethnicity', 'Socioeconomic_Status', 'Parental_Education_Level', 'Disability_Status', 'GPA', 'Past_Academic_Performance', 'Current_Semester_Performance', 'Courses_Failed', 'Credits_Completed', 'Study_Hours_per_Week', 'Attendance_Rate', 'Number_of_Late_Submissions', 'Class_Participation_Score', 'Online_Learning_Hours', 'Library_Usage_Hours', 'Disciplinary_Actions', 'Social_Engagement_Score', 'Mental_Health_Score', 'Extracurricular_Activities', 'Dropout_Risk', 'Dropout_Risk_Label']


In [2]:

# Task 1.1: GPA Histogram with Range Slider

data = df.dropna(subset=["GPA"])

fig1 = px.histogram(
    data,
    x="GPA",
    nbins=20,
    marginal="box",
    title="Distribution of Student GPA",
    labels={
        "GPA": "Grade Point Average",
        "count": "Number of Students"
    }
)

fig1.update_layout(
    xaxis_rangeslider_visible=True,
    xaxis_title="GPA",
    yaxis_title="Number of Students",
    template="plotly_white"
)

fig1.show()


In [4]:

# Task 1.2: Interactive Scatter Plot

data = df.dropna(
    subset=["Study_Hours_per_Week", "GPA"]
)

hover_cols = [
    c for c in [
        "Student_ID",
        "Attendance_Rate",
        "Current_Semester_Performance",
        "Courses_Failed"
    ] if c in data.columns
]

fig2 = px.scatter(
    data,
    x="Study_Hours_per_Week",
    y="GPA",
    color="Dropout_Risk_Label",
    hover_data=hover_cols,
    title="GPA vs Study Hours by Dropout Risk",
    labels={
        "Study_Hours_per_Week": "Study Hours per Week",
        "GPA": "Grade Point Average",
        "Dropout_Risk_Label": "Dropout Risk"
    }
)

fig2.update_layout(template="plotly_white")
fig2.show()


In [5]:

# Task 1.3: Box Plot with All Student Points

data = df.dropna(subset=["GPA"])

fig3 = px.box(
    data,
    x="Dropout_Risk_Label",
    y="GPA",
    color="Dropout_Risk_Label",
    points="all",
    hover_data=[
        c for c in ["Student_ID", "Attendance_Rate"]
        if c in data.columns
    ],
    title="GPA Distribution by Dropout Risk",
    labels={
        "Dropout_Risk_Label": "Dropout Risk",
        "GPA": "Grade Point Average"
    }
)

fig3.update_layout(
    template="plotly_white",
    showlegend=False
)

fig3.show()


In [6]:

# Task 1.4: Interactive 3D Scatter Plot

data = df.dropna(
    subset=[
        "GPA",
        "Attendance_Rate",
        "Current_Semester_Performance"
    ]
)

fig4 = px.scatter_3d(
    data,
    x="GPA",
    y="Attendance_Rate",
    z="Current_Semester_Performance",
    color="Dropout_Risk_Label",
    hover_data=[
        c for c in ["Student_ID", "Study_Hours_per_Week"]
        if c in data.columns
    ],
    title="3D Student Performance Analysis",
    labels={
        "GPA": "GPA",
        "Attendance_Rate": "Attendance Rate",
        "Current_Semester_Performance": "Semester Performance"
    }
)

fig4.update_layout(template="plotly_white")
fig4.show()


In [7]:

# Task 2.1: Animated Scatter Plot by Age

data = df.dropna(
    subset=[
        "Age",
        "GPA",
        "Study_Hours_per_Week",
        "Current_Semester_Performance"
    ]
).copy()

data["Age_Frame"] = data["Age"].astype(int).astype(str)

age_order = [
    str(x) for x in sorted(data["Age"].astype(int).unique())
]

# Fixed axes keep the scale consistent across frames
x_min = data["Study_Hours_per_Week"].min()
x_max = data["Study_Hours_per_Week"].max()
y_min = data["GPA"].min()
y_max = data["GPA"].max()

data["Marker_Size"] = data[
    "Current_Semester_Performance"
].clip(lower=0)

fig5 = px.scatter(
    data,
    x="Study_Hours_per_Week",
    y="GPA",
    animation_frame="Age_Frame",
    animation_group="Student_ID" if "Student_ID" in data.columns else None,
    size="Marker_Size",
    size_max=20,
    color="Dropout_Risk_Label",
    category_orders={"Age_Frame": age_order},
    range_x=[x_min - 1, x_max + 1],
    range_y=[y_min - 0.2, y_max + 0.2],
    hover_data=[
        c for c in [
            "Student_ID",
            "Attendance_Rate",
            "Current_Semester_Performance"
        ] if c in data.columns
    ],
    title="Animated GPA vs Study Hours by Age",
    labels={
        "Study_Hours_per_Week": "Study Hours per Week",
        "GPA": "Grade Point Average",
        "Age_Frame": "Age Group",
        "Marker_Size": "Semester Performance"
    }
)

fig5.update_layout(template="plotly_white")
fig5.show()

# Note: A real year-based animation requires a date/year column.


In [8]:

# Task 2.2: Scatter Plot with Dropdown Menu

metrics = {
    "GPA": "GPA",
    "Attendance Rate": "Attendance_Rate",
    "Semester Performance": "Current_Semester_Performance",
    "Courses Failed": "Courses_Failed"
}

required_cols = [
    "Study_Hours_per_Week",
    *metrics.values()
]

data = df.dropna(subset=required_cols).copy()

fig6 = go.Figure()

# Initial chart: Study Hours vs GPA
fig6.add_trace(
    go.Scatter(
        x=data["Study_Hours_per_Week"],
        y=data["GPA"],
        mode="markers",
        name="Students",
        text=(
            data["Student_ID"].astype(str)
            if "Student_ID" in data.columns
            else None
        ),
        marker=dict(
            size=9,
            color=data["Dropout_Risk"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Dropout Risk")
        ),
        hovertemplate=(
            "Student: %{text}<br>"
            "Study Hours: %{x}<br>"
            "Selected Metric: %{y}<extra></extra>"
        )
    )
)

buttons = []

for label, column in metrics.items():
    buttons.append(
        dict(
            label=label,
            method="update",
            args=[
                {"y": [data[column].tolist()]},
                {
                    "title": f"Study Hours vs {label}",
                    "yaxis": {"title": label}
                }
            ]
        )
    )

fig6.update_layout(
    title="Study Hours vs GPA",
    xaxis_title="Study Hours per Week",
    yaxis_title="GPA",
    template="plotly_white",
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=1.02,
            y=1,
            xanchor="left",
            yanchor="top"
        )
    ],
    margin=dict(r=190)
)

fig6.show()
